# Day 3-4：现代 Tokenizer — BPE 原理与手写实现

> **目标**：理解 BPE/Unigram/SentencePiece 的原理，手写 BPE 编码器，使用 tiktoken 进行实践。

---

## 为什么 Tokenizer 如此重要？

大模型不直接处理文本字符串，而是处理 **token**（整数 ID）。Tokenizer 是文本与模型之间的桥梁：

```
"Hello world" → Tokenizer → [15496, 995] → Embedding → 向量 → 模型
```

Tokenizer 的好坏直接影响：
- **模型效率**：token 数越少，序列越短，计算量越小
- **多语言能力**：差的 tokenizer 会把中文拆成大量 byte，效率极低
- **OOV 问题**：能否处理未见过的词

## 一、Tokenizer 技术路线概览

| 方法 | 代表 | 核心思想 | 使用者 |
|------|------|---------|--------|
| **Word-level** | 传统 NLP | 按空格/标点切分 | 早期模型 |
| **Character-level** | — | 按字符切分 | 极少使用 |
| **BPE** | GPT-2/3/4, LLaMA | 从字符开始，反复合并最高频 pair | 主流方法 |
| **Unigram** | T5, ALBERT | 从大词表开始，逐步删除低概率 token | SentencePiece |
| **WordPiece** | BERT | 类似 BPE，但用似然度而非频率选择合并 | BERT 系列 |

### 现代 LLM 主流：BPE

几乎所有 Decoder-only 大模型都使用 BPE 或其变体：
- GPT-2/3/4: BPE (通过 tiktoken)
- LLaMA: BPE (通过 SentencePiece)
- DeepSeek: BPE
- Qwen: BPE (tiktoken 变体)

## 二、BPE 算法原理（Byte Pair Encoding）

BPE 最初来自数据压缩领域（Gage, 1994），后被 Sennrich et al. (2016) 引入 NLP。

### 核心思想

1. 从最小单元（字符或 byte）开始
2. 统计所有相邻 pair 的出现频率
3. 合并频率最高的 pair 为新 token
4. 重复步骤 2-3，直到达到目标词表大小

### 手动示例

假设训练语料（附带频次）：

```
"low"    : 5 次
"lower"  : 2 次  
"newest" : 6 次
"widest" : 3 次
```

**初始化**（拆成字符 + 词尾标记 `</w>`）：

```
l o w </w>       : 5
l o w e r </w>   : 2
n e w e s t </w> : 6
w i d e s t </w> : 3
```

**Step 1**：统计 pair 频率
- (e, s) → 6 + 3 = 9  ← 最高！
- (s, t) → 6 + 3 = 9  ← 并列
- (l, o) → 5 + 2 = 7
- ...

合并 (e, s) → `es`：

```
l o w </w>        : 5
l o w e r </w>    : 2
n e w es t </w>   : 6
w i d es t </w>   : 3
```

**Step 2**：统计新的 pair 频率
- (es, t) → 6 + 3 = 9  ← 最高！

合并 (es, t) → `est`：

```
l o w </w>        : 5
l o w e r </w>    : 2
n e w est </w>    : 6
w i d est </w>    : 3
```

以此类推...最终得到一个由高频 subword 构成的词表。

## 三、手写 BPE 编码器

下面我们从零实现一个完整的 BPE Tokenizer，包括 **训练** 和 **编解码**。

In [1]:
from collections import Counter, defaultdict
import re


class BPETokenizer:
    """从零实现的 BPE Tokenizer。"""

    def __init__(self, vocab_size: int = 300):
        self.vocab_size = vocab_size
        self.merges: list[tuple[str, str]] = []  # 合并规则列表（按优先级排列）
        self.vocab: dict[str, int] = {}          # token → id
        self.inverse_vocab: dict[int, str] = {}  # id → token

    # ========== 训练阶段 ==========

    def _build_word_freqs(self, text: str) -> dict[tuple[str, ...], int]:
        """将文本按空格分词，每个词拆成字符 tuple，统计频率。
        词尾加 '</w>' 标记，以便区分 "low" 和 "lower" 中的 "low"。
        """
        words = text.strip().split()
        word_counts = Counter(words)
        word_freqs = {}
        for word, freq in word_counts.items():
            symbols = tuple(list(word) + ['</w>'])
            word_freqs[symbols] = freq
        return word_freqs

    def _get_pair_stats(self, word_freqs: dict) -> Counter:
        """统计所有相邻 symbol pair 的频率。"""
        pairs = Counter()
        for symbols, freq in word_freqs.items():
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i + 1])] += freq
        return pairs

    def _merge_pair(self, pair: tuple[str, str], word_freqs: dict) -> dict:
        """在所有词中执行一次合并操作。"""
        new_word_freqs = {}
        bigram = pair
        replacement = pair[0] + pair[1]

        for symbols, freq in word_freqs.items():
            new_symbols = []
            i = 0
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == bigram[0] and symbols[i + 1] == bigram[1]:
                    new_symbols.append(replacement)
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            new_word_freqs[tuple(new_symbols)] = freq

        return new_word_freqs

    def train(self, text: str):
        """在给定文本上训练 BPE。"""
        word_freqs = self._build_word_freqs(text)

        # 初始词表：所有单个字符 + '</w>'
        all_chars = set()
        for symbols in word_freqs.keys():
            for s in symbols:
                all_chars.add(s)

        num_merges = self.vocab_size - len(all_chars)

        print(f"初始字符集大小: {len(all_chars)}")
        print(f"目标词表大小: {self.vocab_size}")
        print(f"需要执行合并次数: {num_merges}")
        print("---")

        for i in range(num_merges):
            pair_stats = self._get_pair_stats(word_freqs)
            if not pair_stats:
                break

            best_pair = pair_stats.most_common(1)[0][0]
            self.merges.append(best_pair)
            word_freqs = self._merge_pair(best_pair, word_freqs)

            if i < 10 or i % 50 == 0:
                print(f"Merge #{i+1}: {best_pair} → '{best_pair[0]+best_pair[1]}' (freq={pair_stats[best_pair]})")

        # 构建最终词表
        final_tokens = set(all_chars)
        for a, b in self.merges:
            final_tokens.add(a + b)

        self.vocab = {token: idx for idx, token in enumerate(sorted(final_tokens))}
        self.inverse_vocab = {idx: token for token, idx in self.vocab.items()}

        print(f"\n训练完成! 最终词表大小: {len(self.vocab)}")

    # ========== 编码阶段 ==========

    def _tokenize_word(self, word: str) -> list[str]:
        """对单个词应用学习到的 BPE 合并规则。"""
        symbols = list(word) + ['</w>']

        for pair in self.merges:
            i = 0
            new_symbols = []
            while i < len(symbols):
                if i < len(symbols) - 1 and symbols[i] == pair[0] and symbols[i + 1] == pair[1]:
                    new_symbols.append(pair[0] + pair[1])
                    i += 2
                else:
                    new_symbols.append(symbols[i])
                    i += 1
            symbols = new_symbols

        return symbols

    def encode(self, text: str) -> list[int]:
        """将文本编码为 token id 序列。"""
        tokens = []
        for word in text.strip().split():
            word_tokens = self._tokenize_word(word)
            for t in word_tokens:
                if t in self.vocab:
                    tokens.append(self.vocab[t])
                else:
                    # OOV: 回退到单字符
                    for ch in t:
                        tokens.append(self.vocab.get(ch, 0))
        return tokens

    def decode(self, ids: list[int]) -> str:
        """将 token id 序列解码回文本。"""
        tokens = [self.inverse_vocab.get(i, '?') for i in ids]
        text = ''.join(tokens)
        text = text.replace('</w>', ' ')
        return text.strip()

### 训练我们的 BPE Tokenizer

In [2]:
# 用一段英文语料训练
corpus = """
the cat sat on the mat the cat ate the rat the dog sat on the log
a low lower lowest the newest and the widest range of products
natural language processing is a subfield of linguistics
deep learning models have transformed natural language processing
transformers use self attention mechanisms to process sequences
large language models can generate coherent text and follow instructions
the transformer architecture uses multi head attention and feed forward layers
byte pair encoding is used to tokenize text in modern language models
the model learns to predict the next token given the previous context
scaling laws show that larger models trained on more data perform better
"""

tokenizer = BPETokenizer(vocab_size=100)
tokenizer.train(corpus)

初始字符集大小: 26
目标词表大小: 100
需要执行合并次数: 74
---
Merge #1: ('e', '</w>') → 'e</w>' (freq=26)
Merge #2: ('s', '</w>') → 's</w>' (freq=18)
Merge #3: ('t', '</w>') → 't</w>' (freq=16)
Merge #4: ('t', 'h') → 'th' (freq=13)
Merge #5: ('a', 'n') → 'an' (freq=13)
Merge #6: ('th', 'e</w>') → 'the</w>' (freq=12)
Merge #7: ('e', 'r') → 'er' (freq=10)
Merge #8: ('d', '</w>') → 'd</w>' (freq=10)
Merge #9: ('n', '</w>') → 'n</w>' (freq=9)
Merge #10: ('o', 'd') → 'od' (freq=8)
Merge #51: ('proc', 'es') → 'proces' (freq=3)

训练完成! 最终词表大小: 100


In [3]:
# 测试编码和解码
test_texts = [
    "the cat sat on the mat",
    "language models are great",
    "natural language processing",
    "the newest model",
]

for text in test_texts:
    encoded = tokenizer.encode(text)
    decoded = tokenizer.decode(encoded)
    tokens = [tokenizer.inverse_vocab[i] for i in encoded]
    print(f"原文: '{text}'")
    print(f"Token: {tokens}")
    print(f"ID:    {encoded}")
    print(f"解码: '{decoded}'")
    print("---")

原文: 'the cat sat on the mat'
Token: ['the</w>', 'cat</w>', 'sat</w>', 'on</w>', 'the</w>', 'm', 'at</w>']
ID:    [82, 12, 76, 64, 82, 50, 8]
解码: 'the cat sat on the mat'
---
原文: 'language models are great'
Token: ['language</w>', 'models</w>', 'ar', 'e</w>', 'g', 'r', 'e', 'at</w>']
ID:    [44, 53, 6, 16, 29, 73, 15, 8]
解码: 'language models are great'
---
原文: 'natural language processing'
Token: ['natural</w>', 'language</w>', 'processing</w>']
ID:    [59, 44, 71]
解码: 'natural language processing'
---
原文: 'the newest model'
Token: ['the</w>', 'ne', 'w', 'est</w>', 'model', '</w>']
ID:    [82, 60, 95, 23, 52, 0]
解码: 'the newest model'
---


### 可视化：查看学到的词表

In [4]:
# 查看前 20 次合并规则
print("前 20 次合并规则:")
print("=" * 40)
for i, (a, b) in enumerate(tokenizer.merges[:20]):
    print(f"  #{i+1}: '{a}' + '{b}' → '{a+b}'")

print(f"\n词表大小: {len(tokenizer.vocab)}")
print(f"\n完整词表 (部分):")
for token, idx in sorted(tokenizer.vocab.items(), key=lambda x: x[1])[:30]:
    print(f"  {idx:3d}: '{token}'")

前 20 次合并规则:
  #1: 'e' + '</w>' → 'e</w>'
  #2: 's' + '</w>' → 's</w>'
  #3: 't' + '</w>' → 't</w>'
  #4: 't' + 'h' → 'th'
  #5: 'a' + 'n' → 'an'
  #6: 'th' + 'e</w>' → 'the</w>'
  #7: 'e' + 'r' → 'er'
  #8: 'd' + '</w>' → 'd</w>'
  #9: 'n' + '</w>' → 'n</w>'
  #10: 'o' + 'd' → 'od'
  #11: 'i' + 'n' → 'in'
  #12: 'a' + 't</w>' → 'at</w>'
  #13: 'a' + 't' → 'at'
  #14: 'g' + '</w>' → 'g</w>'
  #15: 'e' + 'l' → 'el'
  #16: 'e' + 'n' → 'en'
  #17: 'e' + 's' → 'es'
  #18: 'g' + 'e</w>' → 'ge</w>'
  #19: 'p' + 'r' → 'pr'
  #20: 'a' + 'r' → 'ar'

词表大小: 100

完整词表 (部分):
    0: '</w>'
    1: 'a'
    2: 'a</w>'
    3: 'al'
    4: 'an'
    5: 'and</w>'
    6: 'ar'
    7: 'at'
    8: 'at</w>'
    9: 'ate</w>'
   10: 'b'
   11: 'c'
   12: 'cat</w>'
   13: 'd'
   14: 'd</w>'
   15: 'e'
   16: 'e</w>'
   17: 'ed</w>'
   18: 'el'
   19: 'en'
   20: 'er'
   21: 'er</w>'
   22: 'es'
   23: 'est</w>'
   24: 'f'
   25: 'f</w>'
   26: 'fo'
   27: 'for'
   28: 'form'
   29: 'g'


## 四、Byte-level BPE（GPT-2 的做法）

GPT-2 的关键改进：不从字符开始，而是从 **byte**（256 个）开始：

| 方法 | 初始词表 | OOV 问题 | 中文处理 |
|------|---------|---------|----------|
| 字符级 BPE | ~26 字母 + 标点 | 有 | 需要大词表 |
| **Byte-level BPE** | 256 bytes | **完全没有** | 自然支持(UTF-8 bytes) |

这就是为什么 GPT-2/3/4 和 LLaMA 可以处理任何语言、任何符号 —— 因为任何文本最终都是 bytes。

### tiktoken：OpenAI 的高效 BPE 实现

In [ ]:
# 安装 tiktoken: pip install tiktoken
import tiktoken

# GPT-4 使用的 tokenizer
enc = tiktoken.encoding_for_model("gpt-4")

test_texts = [
    "Hello, world!",
    "Transformer is all you need.",
    "大语言模型的核心是注意力机制",  # 中文
    "def hello():\n    print('world')",  # 代码
]

for text in test_texts:
    tokens = enc.encode(text)
    decoded_tokens = [enc.decode([t]) for t in tokens]
    print(f"文本: '{text}'")
    print(f"Token数: {len(tokens)}")
    print(f"Tokens: {decoded_tokens}")
    print(f"IDs: {tokens}")
    print("---")

### 对比不同模型的 Tokenizer

观察同一句中文在不同 tokenizer 下的 token 数量差异 —— 这直接影响模型的中文处理效率。

In [ ]:
# 对比 GPT-2 vs GPT-4 的 tokenizer 效率
enc_gpt2 = tiktoken.encoding_for_model("gpt-3.5-turbo")  # cl100k_base
enc_gpt4 = tiktoken.encoding_for_model("gpt-4")          # cl100k_base

chinese_text = "大语言模型通过预测下一个token来学习语言的规律和模式"

tokens_gpt2 = enc_gpt2.encode(chinese_text)
tokens_gpt4 = enc_gpt4.encode(chinese_text)

print(f"中文文本: '{chinese_text}'")
print(f"字符数: {len(chinese_text)}")
print(f"\nGPT-3.5 (cl100k_base): {len(tokens_gpt2)} tokens")
print(f"GPT-4   (cl100k_base): {len(tokens_gpt4)} tokens")

# 展示每个 token 对应的文本
print(f"\n逐 token 拆解 (GPT-4):")
for t in tokens_gpt4:
    decoded = enc_gpt4.decode([t])
    print(f"  ID={t:6d} → '{decoded}'")

## 五、Embedding 层设计

Token ID 需要通过 Embedding 层转化为稠密向量，才能被模型处理。

```
token_id → Embedding Table → d_model 维向量
[15496]  → E[15496]        → [0.12, -0.34, 0.56, ...] (768维或更高)
```

### 关键设计决策

1. **词表大小 V**：通常 32K ~ 128K
   - GPT-2: 50,257
   - LLaMA: 32,000
   - LLaMA-3: 128,256
   - 更大的词表 → 更短的序列，但 embedding 矩阵更大

2. **嵌入维度 d_model**：
   - GPT-2 Small: 768
   - LLaMA-7B: 4096
   - LLaMA-70B: 8192

3. **权重共享（Weight Tying）**：
   - 许多模型将 embedding 层和最终输出层的权重矩阵共享
   - 即 `output_logits = hidden @ E.T`
   - 节省参数，且语义上合理

In [ ]:
import torch
import torch.nn as nn

# Embedding 层示例
vocab_size = 32000
d_model = 4096

embedding = nn.Embedding(vocab_size, d_model)

# 模拟输入
input_ids = torch.tensor([[1, 234, 5678, 910, 2]])  # batch_size=1, seq_len=5
embedded = embedding(input_ids)

print(f"词表大小: {vocab_size}")
print(f"嵌入维度: {d_model}")
print(f"Embedding 参数量: {vocab_size * d_model / 1e6:.1f}M")
print(f"输入形状: {input_ids.shape}")
print(f"输出形状: {embedded.shape}  # (batch, seq_len, d_model)")

## 六、词表扩展（为第 7 周 Chinese-LLaMA 铺垫）

当一个英文模型需要支持中文时，核心问题是：英文 tokenizer 会把中文拆成大量 byte-level tokens，效率极低。

**解决方案：词表扩展**

1. 在中文语料上训练一个新的 BPE tokenizer
2. 从新词表中选取高频中文 subword
3. 将这些新 token 添加到原始词表
4. 扩展 embedding 矩阵（新 token 用均值/随机初始化）

```python
# 伪代码
old_vocab_size = 32000
new_chinese_tokens = 10000  # 新增中文 token
new_vocab_size = old_vocab_size + new_chinese_tokens  # 42000

# 扩展 embedding
old_embed = model.embedding.weight  # (32000, 4096)
new_embed = torch.randn(new_chinese_tokens, 4096) * 0.02
model.embedding.weight = nn.Parameter(torch.cat([old_embed, new_embed], dim=0))  # (42000, 4096)
```

这就是 Chinese-LLaMA 的核心思路之一，第 7 周会详细实现。

## 七、练习题

### 练习 1：理解 BPE 合并过程
给定语料 `"aab aac aba baa"` 和初始字符集 `{a, b, c, </w>}`，手动执行 3 步 BPE 合并。

### 练习 2：BPE 编码新词
用我们训练好的 BPE tokenizer 编码 `"processing models"` ，验证结果是否合理。

### 练习 3：比较效率
用 tiktoken 分别对 100 字的中文文本和 100 词的英文文本进行编码，比较 token 数量比。思考：为什么中文需要更多 tokens？

### 练习 4（选做）：实现 Unigram Tokenizer
Unigram 的思路与 BPE 相反：从大词表开始，逐步删除。尝试实现一个简化版。